In [ ]:
# Step 1: match_sop(Diagnose incident for Train Ticket microservices. Incident window: 2024-01-23 18:47:25 to 2024-01-23 19:17:25 UTC. Avail
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.43)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.41)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.39)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_comparison = component_analyze(start_time, end_time)
answer = metric_evidence + ' ' + component_comparison

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_comparison = component_analyze(start_time, end_time)
answer = metric_evidence + ' ' + component_comparison
```

In [ ]:
# Step 3: run_sop
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_comparison = component_analyze(start_time, end_time)
answer = metric_evidence + ' ' + component_comparison

```
[ANOMALY] ts-order-service latency-90: rise to 0.1846 at 2024-01-23 18:58:00 (median 0.02265, thr 0.02596, sev 51.0, n=1)
[ANOMALY] ts-auth-service diskio: rise to 1.909e+05 at 2024-01-23 18:50:00 (median 5.236e+04, thr 8.426e+04, sev 21.9, n=2)
[ANOMALY] ts-route-service cpu: rise to 10.44 at 2024-01-23 19:00:00 (median 4.206, thr 7.932, sev 13.2, n=2)
[ANOMALY] ts-auth-service latency-50: rise to 1.862 at 2024-01-23 19:06:00 (median 0.2953, thr 1.795, sev 12.5, n=2)
[ANOMALY] ts-route-service mem: drop to 2.558e+08 at 2024-01-23 18:50:00 (median 2.589e+08, thr 2.558e+08, sev 10.6, n=2)
[ANOMALY] ts-travel-service cpu: rise to 7.598 at 2024-01-23 18:50:00 (median 2.845, thr 5.055, sev 8.6, n=2)
[ANOMALY] ts-route-service latency-50: rise to 0.003946 at 2024-01-23 19:01:00 (median 0.003397, thr 0.003649, sev 6.6, n=1)
[ANOMALY] ts-order-service diskio: drop to 2.198e+04 at 2024-01-23 19:14:00 (median 5.312e+04, thr 3.376e+04, sev 4.8, n=1)
[ANOMALY] ts-route-service latency-90: rise to 0.01004 at 2024-01-23 19:00:00 (median 0.007128, thr 0.009269, sev 4.6, n=2)
[ANOMALY] ts-order-service latency-50: rise to 0.012 at 2024-01-23 19:13:00 (median 0.008335, thr 0.01146, sev 4.2, n=2)
[ANOMALY] ts-auth-service cpu: rise to 39.88 at 2024-01-23 19:06:00 (median 16.81, thr 39.81, sev 3.6, n=2)
[ANOMALY] ts-route-service socket: rise to 17.93 at 2024-01-23 18:50:00 (median 11.72, thr 17.84, sev 3.2, n=2) Component anomaly overview (ranked):
ts-route-service  metric_anoms=5 (top: cpu sev=13.2)  trace_anoms=12  log_anoms=1
ts-travel-service  metric_anoms=1 (top: cpu sev=8.6)  trace_anoms=17  log_anoms=0
ts-order-service  metric_anoms=3 (top: latency-90 sev=51.0)  trace_anoms=7  log_anoms=0
ts-auth-service  metric_anoms=3 (top: diskio sev=21.9)  trace_anoms=5  log_anoms=1
ts-train-service  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=8  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] ts-order-service latency-90: rise to 0.1846 at 2024-01-23 18:58:00 (median 0.02265, thr 0.02596, sev 51.0, n=1
...[truncated])

```
ObAgent: The fault class is likely a socket error.

Reason: ts-route-service shows a discrete rise in the socket metric (17.93 > thr) at 18:50 concurrent with small CPU elevation and multiple latency anomalies downstream (notably ts-order-service latency-90 spike). That pattern (socket errors -> retries/blocking -> higher CPU and increased request latency) fits best. There is no clear retransmit/throughput signature for packet loss, and resource rises are not pervasive enough to indicate a pure CPU/memory/disk fault. Confirm by inspecting route-service connection/error logs and TCP retransmit/error counters.
JudgeAgent: No - The agent produced a hypothesis (ts-route-service = socket error) citing a concrete metric (ts-route-service socket rise to 17.93 at 2024-01-23 18:50:00) and downstream latency spikes, but this was not confirmed. The history lacks evidence that all ts-route-service pods are anomalous (required to pick the service as the location) and no supporting logs/traces/TCP error counters were provided. Therefore the root cause is not fully and definitively pinned.
```

In [ ]:
# Step 5: match_sop(Diagnose socket/connection error in Train Ticket microservices. Incident window: 2024-01-23 18:47:25 to 2024-01-23 19:17
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.38) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.36)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.35)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 6: generate_sop_code for 'RCA-Agent Trace Localization SOP'
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
trace_summary = collect_trace(start_time, end_time, service=None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_summary + ' ' + metric_evidence

```
Generated code for SOP 'RCA-Agent Trace Localization SOP':
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
trace_summary = collect_trace(start_time, end_time, service=None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_summary + ' ' + metric_evidence
```

In [ ]:
# Step 7: run_sop
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
trace_summary = collect_trace(start_time, end_time, service=None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_summary + ' ' + metric_evidence

```
[TRACE] ts-food-map-service trace.from_ts-food-service.duration_p95: rise to 0.07136 at 2024-01-23 18:55:00 (median 0.005227, sev 51.1)
[TRACE] ts-food-service trace.to_ts-food-map-service.duration_p95: rise to 0.07136 at 2024-01-23 18:55:00 (median 0.005227, sev 51.1)
[TRACE] ts-basic-service trace.to_ts-route-service.duration_p95: rise to 0.03591 at 2024-01-23 19:00:00 (median 0.004066, sev 50.5)
[TRACE] ts-route-service trace.from_ts-basic-service.duration_p95: rise to 0.03591 at 2024-01-23 19:00:00 (median 0.004066, sev 50.5)
[TRACE] ts-route-service trace.from_ts-travel-service.duration_p95: rise to 0.01275 at 2024-01-23 19:00:00 (median 0.004189, sev 50.4)
[TRACE] ts-travel-service trace.to_ts-route-service.duration_p95: rise to 0.01275 at 2024-01-23 19:00:00 (median 0.004189, sev 50.4)
[TRACE] ts-contacts-service trace.from_ts-preserve-other-service.duration_p95: rise to 0.05991 at 2024-01-23 18:53:00 (median 0.005774, sev 47.6)
[TRACE] ts-preserve-other-service trace.to_ts-contacts-service.duration_p95: rise to 0.05991 at 2024-01-23 18:53:00 (median 0.005774, sev 47.6)
[TRACE] ts-admin-travel-service trace.to_ts-travel2-service.duration_p95: rise to 0.3791 at 2024-01-23 19:00:00 (median 0.0722, sev 47.4)
[TRACE] ts-travel2-service trace.from_ts-admin-travel-service.duration_p95: rise to 0.3791 at 2024-01-23 19:00:00 (median 0.0722, sev 47.4)
[TRACE] ts-contacts-service trace.from_ts-preserve-service.duration_p95: rise to 0.03548 at 2024-01-23 18:52:00 (median 0.005697, sev 45.7)
[TRACE] ts-preserve-service trace.to_ts-contacts-service.duration_p95: rise to 0.03548 at 2024-01-23 18:52:00 (median 0.005697, sev 45.7)
[TRACE] ts-basic-service trace.to_ts-route-service.duration_mean: rise to 0.006856 at 2024-01-23 18:54:00 (median 0.003375, sev 45.1)
[TRACE] ts-route-service trace.from_ts-basic-service.duration_mean: rise to 0.006856 at 2024-01-23 18:54:00 (median 0.003375, sev 45.1)
[TRACE] ts-admin-basic-info-service trace.to_ts-price-service.duration_p95: rise to 0.02494 at 2024-01-23 18:51:00 (median 0.00376, sev 43.4)
Most-downstream faulty component(s): root, ts-assurance-service, ts-auth-service, ts-config-service, ts-consign-price-service, ts-contacts-service, ts-food-map-service, ts-payment-service, ts-price-service, ts-route-service, ts-station-service, ts-train-service, ts-user-service [ANOMALY] ts-order-service latency-90: rise to 0.1846 at 2024-01-23 18:58:00 (median 0.02265, thr 0.02596, sev 51.0, n=1)
[ANOMALY] ts-auth-service diskio: rise to 1.909e+05 at 2024-01-23 18:50:00 (median 5.236e+04, thr 8.426e+04, sev 21.9, n=2)
[ANOMALY] ts-route-service cpu: rise to 10.44 at 2024-01-23 19:00:00 (median 4.206, thr 7.932, sev 13.2, n=2)
[ANOMALY] ts-auth-service latency-50: rise to 1.862 at 2024-01-23 19:06:00 (median 0.2953, thr 1.795, sev 12.5, n=2)
[ANOMALY] ts-route-service mem: drop to 2.558e+08 at 2024-01-23 18:50:00 (median 2.589e+08, thr 2.558e+08, sev 10.6, n=2)
[ANOMALY] ts-travel-service cpu: rise to 7.598 at 2024-01-23 18:50:00 (median 2.845, thr 5.055, sev 8.6, n=2)
[ANOMALY] ts-route-service latency-50: rise to 0.003946 at 2024-01-23 19:01:00 (median 0.003397, thr 0.003649, sev 6.6, n=1)
[ANOMALY] ts-order-service diskio: drop to 2.198e+04 at 2024-01-23 19:14:00 (median 5.312e+04, thr 3.376e+04, sev 4.8, n=1)
[ANOMALY] ts-route-service latency-90: rise to 0.01004 at 2024-01-23 19:00:00 (median 0.007128, thr 0.009269, sev 4.6, n=2)
[ANOMALY] ts-order-service latency-50: rise to 0.012 at 2024-01-23 19:13:00 (median 0.008335, thr 0.01146, sev 4.2, n=2)
[ANOMALY] ts-auth-service cpu: rise to 39.88 at 2024-01-23 19:06:00 (median 16.81, thr 39.81, sev 3.6, n=2)
[ANOMALY] ts-route-service socket: rise to 17.93 at 2024-01-23 18:50:00 (median 11.72, thr 17.84, sev 3.2, n=2)
```

In [ ]:
# Step 8: match_observation([TRACE] ts-food-map-service trace.from_ts-food-service.duration_p95: rise to 0.07136 at 2024-01-23 18:55:00 (median 0.00
...[truncated])

```
ObAgent: The fault class is likely to be network latency.

Analysis: multiple inter-service trace p95s (many service-to-service durations) spiked around 18:55–19:00 and higher-level latency metrics (e.g., ts-order-service latency-90) rose sharply at the same time, indicating increased request/response durations across the mesh. Resource anomalies (diskio/cpu) are either isolated (auth diskio) or smaller/localized, so they don't explain the widespread cross-service latency increase.
JudgeAgent: No - History contains plausible hypotheses (ts-route-service socket error supported by ts-route-service socket rise to 17.93 at 2024-01-23 18:50:00 and downstream latency spikes; and a network-latency hypothesis supported by many service-to-service trace p95 spikes around 18:55–19:00 such as ts-food-map-service trace.from_ts-food-service.duration_p95 rise to 0.07136 and ts-order-service latency-90 rise to 0.1846). However no single component was definitively chosen with confirmation that all its pods were anomalous, and the socket hypothesis was explicitly judged unconfirmed (missing pod-wide evidence, logs/TCP error counters). Therefore the location+type+concrete evidence required for a complete root cause was not all pinned down.
```

In [ ]:
# Step 9: match_sop(Diagnose network latency in Train Ticket microservices. Incident window: 2024-01-23 18:47:25 to 2024-01-23 19:17:25 UTC.
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.33) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.30) (already executed)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.28)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 10: generate_sop_code for 'RCA-Agent Log-Based Reason Analysis SOP'
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
logs = get_logs(None, start_time, end_time)
relevant_metrics = get_relevant_metric('error')
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = logs + ' ' + relevant_metrics + ' ' + metric_evidence

```
Generated code for SOP 'RCA-Agent Log-Based Reason Analysis SOP':
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
logs = get_logs(None, start_time, end_time)
relevant_metrics = get_relevant_metric('error')
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = logs + ' ' + relevant_metrics + ' ' + metric_evidence
```

In [ ]:
# Step 11: run_sop
start_time = '2024-01-23 18:47:25'
end_time = '2024-01-23 19:17:25'
logs = get_logs(None, start_time, end_time)
relevant_metrics = get_relevant_metric('error')
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = logs + ' ' + relevant_metrics + ' ' + metric_evidence

```
[LOG] ts-auth-service log.row_count: drop to 142 at 2024-01-23 18:50:00 (median 308)
[LOG] ts-route-service log.row_count: drop to 399 at 2024-01-23 18:50:00 (median 978) Metric KPIs matching 'error' (top 3):
- error
- workload
- mem [ANOMALY] ts-order-service latency-90: rise to 0.1846 at 2024-01-23 18:58:00 (median 0.02265, thr 0.02596, sev 51.0, n=1)
[ANOMALY] ts-auth-service diskio: rise to 1.909e+05 at 2024-01-23 18:50:00 (median 5.236e+04, thr 8.426e+04, sev 21.9, n=2)
[ANOMALY] ts-route-service cpu: rise to 10.44 at 2024-01-23 19:00:00 (median 4.206, thr 7.932, sev 13.2, n=2)
[ANOMALY] ts-auth-service latency-50: rise to 1.862 at 2024-01-23 19:06:00 (median 0.2953, thr 1.795, sev 12.5, n=2)
[ANOMALY] ts-route-service mem: drop to 2.558e+08 at 2024-01-23 18:50:00 (median 2.589e+08, thr 2.558e+08, sev 10.6, n=2)
[ANOMALY] ts-travel-service cpu: rise to 7.598 at 2024-01-23 18:50:00 (median 2.845, thr 5.055, sev 8.6, n=2)
[ANOMALY] ts-route-service latency-50: rise to 0.003946 at 2024-01-23 19:01:00 (median 0.003397, thr 0.003649, sev 6.6, n=1)
[ANOMALY] ts-order-service diskio: drop to 2.198e+04 at 2024-01-23 19:14:00 (median 5.312e+04, thr 3.376e+04, sev 4.8, n=1)
[ANOMALY] ts-route-service latency-90: rise to 0.01004 at 2024-01-23 19:00:00 (median 0.007128, thr 0.009269, sev 4.6, n=2)
[ANOMALY] ts-order-service latency-50: rise to 0.012 at 2024-01-23 19:13:00 (median 0.008335, thr 0.01146, sev 4.2, n=2)
[ANOMALY] ts-auth-service cpu: rise to 39.88 at 2024-01-23 19:06:00 (median 16.81, thr 39.81, sev 3.6, n=2)
[ANOMALY] ts-route-service socket: rise to 17.93 at 2024-01-23 18:50:00 (median 11.72, thr 17.84, sev 3.2, n=2)
```

In [ ]:
# Step 12: match_observation([LOG] ts-auth-service log.row_count: drop to 142 at 2024-01-23 18:50:00 (median 308)
[LOG] ts-route-service log.row_coun
...[truncated])

```
ObAgent: The fault class is likely to be high CPU usage.

Reason: multiple services show concurrent CPU rises (ts-auth to 39.88, ts-route to 10.44, ts-travel to 7.6) alongside increased latencies and reduced request row_counts, consistent with CPU saturation reducing throughput. The disk I/O spike is localized to ts-auth and looks like a secondary effect rather than the system-wide driver.
JudgeAgent: No - Not pinned: history contains multiple plausible hypotheses (ts-route-service socket error; widespread network latency; high CPU on several services) but no single component was confirmed (no evidence that all pods of a chosen service were anomalous) and no single anomaly type was uniquely supported by the required concrete telemetry (missing pod-wide log/TCP error counters or definitive trace/metric evidence). Therefore the root cause (one component + one type + concrete telemetry) is not fully established.
```